In [20]:
import pandas as pd
import numpy as np

In [21]:
unemployment = pd.read_csv("../data/raw/UNRATE.csv", parse_dates=["date"])
yield_curve = pd.read_csv("../data/raw/T10Y2Y.csv", parse_dates=["date"])
fed_funds = pd.read_csv("../data/raw/FEDFUNDS.csv", parse_dates=["date"])
cpi = pd.read_csv("../data/raw/CPIAUCSL.csv", parse_dates=["date"])
industrial = pd.read_csv("../data/raw/INDPRO.csv", parse_dates=["date"])
housing = pd.read_csv("../data/raw/HOUST.csv", parse_dates=["date"])
claims = pd.read_csv("../data/raw/ICSA.csv", parse_dates=["date"])
sentiment = pd.read_csv("../data/raw/UMCSENT.csv", parse_dates=["date"])
recession = pd.read_csv("../data/raw/USREC.csv", parse_dates=["date"])

In [22]:
datasets = {
    "unemployment": unemployment,
    "yield_curve": yield_curve,
    "fed_funds": fed_funds,
    "cpi": cpi,
    "industrial": industrial,
    "housing": housing,
    "claims": claims,
    "sentiment": sentiment,
    "recession": recession
}

for name, df in datasets.items():
    print(name, df.shape)

unemployment (439, 2)
yield_curve (9551, 2)
fed_funds (439, 2)
cpi (438, 2)
industrial (438, 2)
housing (438, 2)
claims (1909, 2)
sentiment (438, 2)
recession (439, 2)


In [23]:
yield_curve_monthly = (
    yield_curve
    .set_index("date")
    .resample("MS")
    .mean()
    .reset_index()
)

yield_curve_monthly.head()

,date,T10Y2Y
0,1990-01-01,0.121429
1,1990-02-01,0.102632
2,1990-03-01,-0.038182
3,1990-04-01,0.061500
4,1990-05-01,0.115909


In [24]:
claims_monthly = (
    claims
    .set_index("date")
    .resample("MS")
    .mean()
    .reset_index()
)

claims_monthly.head()

,date,ICSA
0,1990-01-01,361000.0
1,1990-02-01,358250.0
2,1990-03-01,345200.0
3,1990-04-01,361750.0
4,1990-05-01,355250.0


In [25]:
print(yield_curve_monthly.shape)
print(claims_monthly.shape)

(440, 2)
(440, 2)


In [26]:
macro = unemployment.copy()

monthly_datasets = [
    yield_curve_monthly,
    fed_funds,
    cpi,
    industrial,
    housing,
    claims_monthly,
    sentiment,
    recession
]

for df in monthly_datasets:
    macro = pd.merge(
        macro,
        df,
        on="date",
        how="left"
    )

macro.head() 

,date,UNRATE,T10Y2Y,FEDFUNDS,CPIAUCSL,INDPRO,HOUST,ICSA,UMCSENT,USREC
0,1990-01-01,5.4,0.121429,8.23,127.5,61.7290,1551.0,361000.0,93.0,0
1,1990-02-01,5.3,0.102632,8.24,128.0,62.2896,1437.0,358250.0,89.5,0
2,1990-03-01,5.2,-0.038182,8.28,128.6,62.5999,1289.0,345200.0,91.3,0
3,1990-04-01,5.4,0.061500,8.26,128.9,62.4359,1248.0,361750.0,93.9,0
4,1990-05-01,5.4,0.115909,8.18,129.1,62.6258,1212.0,355250.0,90.6,0


In [27]:
macro = macro.rename(columns={
    "UNRATE": "unemployment",
    "T10Y2Y": "yield_curve",
    "FEDFUNDS": "fed_funds_rate",
    "CPIAUCSL": "cpi",
    "INDPRO": "industrial_production",
    "HOUST": "housing_starts",
    "ICSA": "initial_claims",
    "UMCSENT": "consumer_sentiment",
    "USREC": "recession"
})

macro.head()

,date,unemployment,yield_curve,fed_funds_rate,cpi,industrial_production,housing_starts,initial_claims,consumer_sentiment,recession
0,1990-01-01,5.4,0.121429,8.23,127.5,61.7290,1551.0,361000.0,93.0,0
1,1990-02-01,5.3,0.102632,8.24,128.0,62.2896,1437.0,358250.0,89.5,0
2,1990-03-01,5.2,-0.038182,8.28,128.6,62.5999,1289.0,345200.0,91.3,0
3,1990-04-01,5.4,0.061500,8.26,128.9,62.4359,1248.0,361750.0,93.9,0
4,1990-05-01,5.4,0.115909,8.18,129.1,62.6258,1212.0,355250.0,90.6,0


In [28]:
macro.isna().sum()

date                     0
unemployment             1
yield_curve              0
fed_funds_rate           0
cpi                      2
industrial_production    1
housing_starts           1
initial_claims           0
consumer_sentiment       1
recession                0
dtype: int64

In [29]:
macro[macro.isna().any(axis=1)]

,date,unemployment,yield_curve,fed_funds_rate,cpi,industrial_production,housing_starts,initial_claims,consumer_sentiment,recession
429,2025-10-01,NaN,0.540455,4.09,NaN,101.2195,1273.0,226750.0,53.6,0
438,2026-07-01,4.1,0.376818,3.63,NaN,NaN,NaN,203250.0,NaN,0


In [30]:
macro.loc[427:431]

,date,unemployment,yield_curve,fed_funds_rate,cpi,industrial_production,housing_starts,initial_claims,consumer_sentiment,recession
427,2025-08-01,4.3,0.560952,4.33,323.291,101.6247,1291.0,229600.0,58.2,0
428,2025-09-01,4.4,0.551905,4.22,324.245,101.6680,1319.0,234000.0,55.1,0
429,2025-10-01,NaN,0.540455,4.09,NaN,101.2195,1273.0,226750.0,53.6,0
430,2025-11-01,4.5,0.543889,3.88,325.063,101.0344,1319.0,222400.0,51.0,0
431,2025-12-01,4.4,0.642273,3.72,326.031,101.4941,1378.0,219250.0,52.9,0


In [31]:
macro.tail(5)

,date,unemployment,yield_curve,fed_funds_rate,cpi,industrial_production,housing_starts,initial_claims,consumer_sentiment,recession
434,2026-03-01,4.3,0.531364,3.64,330.293,101.6172,1522.0,208000.0,53.3,0
435,2026-04-01,4.3,0.520000,3.64,332.407,102.4196,1414.0,207750.0,49.8,0
436,2026-05-01,4.3,0.489000,3.63,333.979,102.5606,1199.0,211600.0,44.8,0
437,2026-06-01,4.2,0.357619,3.63,332.568,102.6395,1427.0,222500.0,49.5,0
438,2026-07-01,4.1,0.376818,3.63,NaN,NaN,NaN,203250.0,NaN,0


In [32]:
macro = macro[macro["date"] < "2026-07-01"].copy()

In [33]:
macro.tail()

,date,unemployment,yield_curve,fed_funds_rate,cpi,industrial_production,housing_starts,initial_claims,consumer_sentiment,recession
433,2026-02-01,4.4,0.654211,3.64,327.460,101.9263,1346.0,215750.0,56.6,0
434,2026-03-01,4.3,0.531364,3.64,330.293,101.6172,1522.0,208000.0,53.3,0
435,2026-04-01,4.3,0.520000,3.64,332.407,102.4196,1414.0,207750.0,49.8,0
436,2026-05-01,4.3,0.489000,3.63,333.979,102.5606,1199.0,211600.0,44.8,0
437,2026-06-01,4.2,0.357619,3.63,332.568,102.6395,1427.0,222500.0,49.5,0


In [34]:
macro.isna().sum()

date                     0
unemployment             1
yield_curve              0
fed_funds_rate           0
cpi                      1
industrial_production    0
housing_starts           0
initial_claims           0
consumer_sentiment       0
recession                0
dtype: int64

In [35]:
continuous_columns = [
    "unemployment",
    "yield_curve",
    "fed_funds_rate",
    "cpi",
    "industrial_production",
    "housing_starts",
    "initial_claims",
    "consumer_sentiment"
]

macro[continuous_columns] = (
    macro[continuous_columns]
    .interpolate(method="linear")
)

In [36]:
macro.isna().sum()

date                     0
unemployment             0
yield_curve              0
fed_funds_rate           0
cpi                      0
industrial_production    0
housing_starts           0
initial_claims           0
consumer_sentiment       0
recession                0
dtype: int64

In [37]:
macro.to_csv(
    "../data/processed/macro_monthly.csv",
    index=False
)